<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-07-buy-accuracy-with-compute.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7 (graded) — Buy accuracy with compute
**Course 2: Generative AI and LLMs with Python — Chapter 7: Reasoning & inference-time compute**

**What you'll submit:** self-consistency and best-of-N applied to a task a small model gets
wrong some of the time, and the accuracy-vs-samples-vs-cost curve with a recommendation.

## 1. Data: real GSM8K math word problems (with offline fallback)

In [ ]:
import re

def load_gsm8k(n=20):
    try:
        from datasets import load_dataset
        ds = load_dataset('gsm8k', 'main', split=f'test[:{n}]')
        problems = []
        for ex in ds:
            final_answer = ex['answer'].split('####')[-1].strip().replace(',', '')
            problems.append({'question': ex['question'], 'answer': float(final_answer)})
        print(f'Loaded {len(problems)} real GSM8K problems.')
        return problems
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — a small set of simple word problems.')
        return [
            {'question': 'A store has 24 apples. It sells 9 and receives a delivery of 15 more. How many apples does it have now?', 'answer': 30.0},
            {'question': 'A train travels 60 miles in 2 hours, then 90 miles in 3 hours. What is its total distance?', 'answer': 150.0},
            {'question': 'A recipe needs 3 eggs per batch. How many eggs for 7 batches?', 'answer': 21.0},
            {'question': 'A worker earns $18/hour and works 6 hours. What are their earnings?', 'answer': 108.0},
        ] * 5

problems = load_gsm8k()

## 2. The model call
Uses your hosted API (Colab Secrets) if configured. **If no key is set, this falls back to a
simulated stochastic reasoner** — a function with a tunable per-sample correctness
probability, standing in for a real small model's genuine sampling variance on hard
problems, so the self-consistency mechanism is still demonstrably correct with zero cost.

In [ ]:
import os, random, re
random.seed(0)

try:
    from google.colab import userdata
    API_KEY = userdata.get('LLM_API_KEY'); BASE_URL = userdata.get('LLM_BASE_URL')
except Exception:
    API_KEY = os.environ.get('LLM_API_KEY'); BASE_URL = os.environ.get('LLM_BASE_URL')
hosted_available = bool(API_KEY and BASE_URL)

def extract_number(text):
    matches = re.findall(r'-?\d+\.?\d*', text.replace(',', ''))
    return float(matches[-1]) if matches else None

def sample_hosted(question, temperature=0.7):
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
    resp = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': question + '\nThink step by step, then give the final numeric answer.'}],
        max_tokens=200, temperature=temperature,
    )
    return resp.choices[0].message.content, resp.usage.total_tokens

def sample_simulated(true_answer, per_sample_accuracy=0.55):
    """A stand-in for a real small model's stochastic reasoning: correct with the given
    probability, otherwise a plausible-looking wrong answer."""
    if random.random() < per_sample_accuracy:
        return true_answer, 120
    return true_answer + random.choice([-10, -5, -2, -1, 1, 2, 5, 10]) * (1 + random.random()), 120

def sample_answer(problem, temperature=0.7):
    if hosted_available:
        text, tokens = sample_hosted(problem['question'], temperature)
        return extract_number(text), tokens
    return sample_simulated(problem['answer']), 120

## 3. Self-consistency: sample N times, take the majority answer

In [ ]:
from collections import Counter

def self_consistency_answer(problem, n_samples):
    answers, total_tokens = [], 0
    for _ in range(n_samples):
        if hosted_available:
            ans, tok = sample_answer(problem)
        else:
            ans, tok = sample_simulated(problem['answer'])
        if ans is not None:
            answers.append(round(ans, 1))
        total_tokens += tok
    if not answers:
        return None, total_tokens
    majority = Counter(answers).most_common(1)[0][0]
    return majority, total_tokens

def evaluate(sample_counts):
    results = []
    for n in sample_counts:
        correct, total_tokens = 0, 0
        for p in problems:
            pred, tok = self_consistency_answer(p, n)
            total_tokens += tok
            if pred is not None and abs(pred - p['answer']) < 0.5:
                correct += 1
        acc = correct / len(problems)
        avg_tokens = total_tokens / len(problems)
        results.append({'n_samples': n, 'accuracy': acc, 'avg_tokens_per_problem': avg_tokens})
        print(f'n_samples={n:2d}  accuracy={acc:.2%}  avg_tokens/problem={avg_tokens:.0f}')
    return results

sample_counts = [1, 3, 5, 9]
curve = evaluate(sample_counts)

## 4. The accuracy / cost curve

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

curve_df = pd.DataFrame(curve)
COST_PER_1K_TOKENS = 0.0002  # placeholder hosted-API rate; use your own provider's rate
curve_df['est_cost_per_problem_usd'] = curve_df['avg_tokens_per_problem'] / 1000 * COST_PER_1K_TOKENS

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(curve_df['n_samples'], curve_df['accuracy'], marker='o', color='tab:blue')
ax1.set_xlabel('samples per question (N)'); ax1.set_ylabel('accuracy', color='tab:blue')
ax2 = ax1.twinx()
ax2.plot(curve_df['n_samples'], curve_df['est_cost_per_problem_usd'], marker='s', color='tab:orange')
ax2.set_ylabel('est. cost per problem ($)', color='tab:orange')
plt.title('Self-consistency: accuracy vs. cost as N grows')
plt.show()
curve_df

## 5. Recommendation (fill in)
At what N does accuracy stop improving meaningfully for the extra cost? Would you recommend
self-consistency for a real-time user-facing feature, or only for a batch/offline job — and
why does that distinction matter here?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 7: Reasoning & inference-time compute*